# Minimum Distance Classifiers

Consider the cork stoppers.xls data set containing measurements performed automatically by an image processing system on 150 cork stoppers belonging to three classes ($\omega_1$ - Super, $\omega_2$ - Average and $\omega_3$ - Poor).<p>

In [1]:
import plotly.graph_objects as go
import numpy as np
import pandas as pd
import os

datasets_folder = os.path.abspath(os.path.join(os.getcwd() ,"../..")) + "/Datasets/" #Set the path to the datasets folder, which is assumed to be two levels up from the current working directory and then inside a "Datasets" folder. Adjust this path as necessary based on your actual directory structure.

xlsFile = pd.read_excel(datasets_folder + "CORK_STOPPERS.XLS",
                        sheet_name="Data",
                        usecols=["C","ART","N.1","PRT","ARM","PRM","ARTG","NG","PRTG","RAAR","RAN"])
CorkStopp = pd.DataFrame(data=xlsFile.dropna()) # remove first empty row 

fnames = CorkStopp.columns[1:] # feature names, excluding the first column which is the class label 'C'

ixSuper = np.where(CorkStopp['C']==1) 
ixAverage = np.where(CorkStopp['C']==2)
ixPoor = np.where(CorkStopp['C']==3)

In your exercise consider only the classes $\omega_1$ and $\omega_2$ and a two-dimensional input space defined by N and PRT/10.<p>
-Find the Euclidean Linear Discriminants $g1(x) and g2(x)?<p>

In [ ]:
# Extract the 'N.1' and 'PRT' features for the "Super" and "Average" classes using the indices obtained from the previous step
NSup = CorkStopp['N.1'].to_numpy()[ixSuper]
NAvg = CorkStopp['N.1'].to_numpy()[ixAverage]

PRT10Sup =(CorkStopp['PRT'].to_numpy()/10)[ixSuper] # create a new variable PRT10Sup by taking the 'PRT' column from the CorkStopp DataFrame, converting it to a NumPy array, dividing it by 10, and then selecting only the elements corresponding to the "Super" class using the indices in ixSuper
PRT10Avg = (CorkStopp['PRT'].to_numpy()/10)[ixAverage] # create a new variable PRT10Avg by taking the 'PRT' column from the CorkStopp DataFrame, converting it to a NumPy array, dividing it by 10, and then selecting only the elements corresponding to the "Average" class using the indices in ixAverage

muSup = np.array([[np.mean(NSup),np.mean(PRT10Sup)]]).T # mean vector for the "Super" class, transposed to be a column vector
muAvg = np.array([[np.mean(NAvg),np.mean(PRT10Avg)]]).T # mean vector for the "Average" class, transposed to be a column vector

print("gSup(x)="+str(muSup.T)+"x-0.5"+str(muSup.T@muSup)) # print the linear discriminant function for the "Super" class, which is a linear function of x with coefficients given by the mean vector and a constant term that depends on the mean vector
print("gAvg(x)="+str(muAvg.T)+"x-0.5"+str(muAvg.T@muAvg)) # print the linear discriminant function for the "Average" class, which is a linear function of x with coefficients given by the mean vector and a constant term that depends on the mean vector


gSup(x)=[[55.28  36.504]]x-0.5[[4388.420416]]
gAvg(x)=[[79.74  64.284]]x-0.5[[10490.900256]]


-Find the equation of the Euclidian MDC decision function, i.e., d(x)=g1(x)-g2(x)?

In [3]:
print("d(x)="+str((muSup-muAvg).T)+"x-0.5"+str((muSup-muAvg).T@(muSup+muAvg)))

d(x)=[[-24.46 -27.78]]x-0.5[[-6102.47984]]


-Compute performance measures?

In [4]:
X1 = np.array([NSup,PRT10Sup]).T
X2 = np.array([NAvg,PRT10Avg]).T
X = np.concatenate((X1,X2),axis=0) # create a new dataset X by concatenating the feature vectors for the "Super" and "Average" classes along the first axis (rows), resulting in a dataset with all samples from both classes
y = np.concatenate((CorkStopp['C'].to_numpy()[ixSuper],CorkStopp['C'].to_numpy()[ixAverage])) # create a new label vector y by concatenating the class labels for the "Super" and "Average" classes

yp = np.ones(np.shape(y)) # vector of predicted labels, initialized to 1 

dx = ((muSup-muAvg).T @ (X.T-0.5*(muSup+muAvg))).flatten() # d(i,j) = gi(x) - gj(x) - > if d(i,j) > 0 then assign to class i, else assign to class j
yp[dx<0]=2

# Calculate the performance metrics based on the true labels y and the predicted labels yp
Hits = np.shape(np.where((y==yp))[0])[0]
TP = np.shape(np.where((y[ixSuper]==yp[ixSuper]))[0])[0]
TN = np.shape(np.where((y[ixAverage]==yp[ixAverage]))[0])[0]
FP = np.shape(np.where((y[ixAverage]!=yp[ixAverage]))[0])[0]
FN = np.shape(np.where((y[ixSuper]!=yp[ixSuper]))[0])[0]

SS = TP/(TP+FN)
SP = TN/(TN+FP)
PR = TP/(TP+FP)
F1Score = 2*(PR*SS)/(PR+SS)
AC = (TN+TP)/(TP+TN+FP+FN)


print("Sensitivity(%)="+str(SS*100))
print("Specificity(%)="+str(SP*100))
print("Precision(%)="+str(PR*100))
print("F1Score(%)="+str(F1Score*100))
print("Accuracy(%)="+str(AC*100))



Sensitivity(%)=84.0
Specificity(%)=78.0
Precision(%)=79.24528301886792
F1Score(%)=81.55339805825243
Accuracy(%)=81.0


-Represent geometrically the samples, class means, and decision hyperplane?

In [5]:
fig=go.Figure()
fig.add_trace(go.Scatter(x=NSup,y=PRT10Sup,name='Super'))
#fig.update_traces(marker=dict(color='green', size=10))
fig.add_trace(go.Scatter(x=NAvg,y=PRT10Avg,name='Average'))
fig.update_traces(marker=dict(size=10))


ixFP = np.where((y[ixAverage]!=yp[ixAverage]))[0]+50
ixFN = np.where((y[ixSuper]!=yp[ixSuper]))[0]

fig.add_trace(go.Scatter(x=X[ixFN,0],y=X[ixFN,1],name='False Average',marker_size=15,
                         marker_symbol="circle-open",marker=dict(color="red",
                                                                 line=dict(width=3,color='red'))))
fig.add_trace(go.Scatter(x=X[ixFP,0],y=X[ixFP,1],name='False Super',marker_size=15,
                         marker_symbol="circle-open",marker=dict(color="blue",
                                                                 line=dict(width=3,color='blue'))))


fig.add_trace(go.Scatter(x=muSup[0],y=muSup[1], marker_size=20,
                         marker_symbol='x',name='Super mean',
                         marker=dict(color="green",line=dict(width=1,
                                        color='black'))))
fig.add_trace(go.Scatter(x=muAvg[0],y=muAvg[1], marker_size=20,
                         marker_symbol='x',name='Average mean',
                         marker=dict(color="yellow",line=dict(width=1,
                                        color='black'))))
fig.add_trace(go.Scatter(x=(muSup[0]+muAvg[0])/2,y=(muSup[1]+muAvg[1])/2, marker_size=20,
                         marker_symbol="circle-dot",name='Mean of the means',
                         marker=dict(color="purple",line=dict(width=1,
                                        color='black'))))

fig.update_traces(mode='markers')
fig.add_trace(go.Scatter(x=[muSup[0][0],muAvg[0][0]],y=[muSup[1][0],muAvg[1][0]],
                         mode='lines', line=dict(color="black",width=4),name='Inter mean segment'))


W = (muSup-muAvg)
b = -0.5*(muSup-muAvg).T@(muSup+muAvg)

x1=np.arange(20,130,1)
#x2=-(W[0,0]/W[1,0])*x1-b/W[1,0]

x2 = - ((muSup[0][0] - muAvg[0][0]) / (muSup[1][0] - muAvg[1][0])) * x1 + 0.5 * (muSup[0][0]**2 + muSup[1][0]**2 - muAvg[0][0]**2 - muAvg[1][0]**2) / (muSup[1][0] - muAvg[1][0])

fig.add_trace(go.Scatter(x=x1,y=x2.flatten(),
                         mode='lines', line=dict(dash='dash',color="gray",width=4),name='Hyperplane'))





fig.update_xaxes(title_text='N')
fig.update_yaxes(title_text='PRT/10')
fig.update_layout(
    autosize=False,
    width=900,
    height=800)
fig.show()

-Find the Mahalanobis Linear Discriminants g1(x) and g2(x)?<p>

In [6]:
NSup = CorkStopp['N.1'].to_numpy()[ixSuper]
NAvg = CorkStopp['N.1'].to_numpy()[ixAverage]

PRT10Sup = (CorkStopp['PRT'].to_numpy()/10)[ixSuper]
PRT10Avg = (CorkStopp['PRT'].to_numpy()/10)[ixAverage]


muSup = np.array([[np.mean(NSup),np.mean(PRT10Sup)]]).T
muAvg = np.array([[np.mean(NAvg),np.mean(PRT10Avg)]]).T

# compute the covariance matrices for the "Super" and "Average" classes 
CSup = np.cov(np.array([NSup,PRT10Sup]))
CAvg = np.cov(np.array([NAvg,PRT10Avg]))

C = (CSup+CAvg)/2
Ci = np.linalg.inv(C)
print(Ci)
            
print("gSup(x)="+str(muSup.T@Ci)+"x-0.5"+str(muSup.T@Ci@muSup))
print("gAvg(x)="+str(muAvg.T@Ci)+"x-0.5"+str(muAvg.T@Ci@muAvg))

[[ 0.0216046  -0.02555068]
 [-0.02555068  0.03601284]]
gSup(x)=[[ 0.26160002 -0.09782884]]x-0.5[[10.89010522]]
gAvg(x)=[[0.08025053 0.27763826]]x-0.5[[24.24687549]]


-Find the equation of the Mahalanobis MDC decision function, i.e., d(x)=g1(x)-g2(x)?

In [7]:
print("d(x)="+str((muSup-muAvg).T@Ci)+"x-0.5"+str((muSup-muAvg).T@Ci@(muSup+muAvg))) #

d(x)=[[ 0.18134949 -0.37546711]]x-0.5[[-13.35677027]]


-Compute performance metrics?

In [8]:
X1=np.array([NSup,PRT10Sup]).T
X2=np.array([NAvg,PRT10Avg]).T
X=np.concatenate((X1,X2),axis=0)
y=np.concatenate((CorkStopp['C'].to_numpy()[ixSuper],CorkStopp['C'].to_numpy()[ixAverage]))

yp=np.ones(np.shape(y))

dx=((muSup-muAvg).T@Ci@(X.T-0.5*(muSup+muAvg))).flatten() # d(i,j) = gi(x) - gj(x) - > if d(i,j) > 0 then assign to class i, else assign to class j
yp[dx<0]=2

Hits=np.shape(np.where((y==yp))[0])[0]
TP=np.shape(np.where((y[ixSuper]==yp[ixSuper]))[0])[0]
TN=np.shape(np.where((y[ixAverage]==yp[ixAverage]))[0])[0]
FP=np.shape(np.where((y[ixAverage]!=yp[ixAverage]))[0])[0]
FN=np.shape(np.where((y[ixSuper]!=yp[ixSuper]))[0])[0]

SS=TP/(TP+FN)
SP=TN/(TN+FP)
PR=TP/(TP+FP)
F1Score=2*(PR*SS)/(PR+SS)
AC=(TN+TP)/(TP+TN+FP+FN)


print("Sensitivity(%)="+str(SS*100))
print("Specificity(%)="+str(SP*100))
print("Precision(%)="+str(PR*100))
print("F1Score(%)="+str(F1Score*100))
print("Accuracy(%)="+str(AC*100))



Sensitivity(%)=98.0
Specificity(%)=82.0
Precision(%)=84.48275862068965
F1Score(%)=90.74074074074075
Accuracy(%)=90.0


-Represent geometrically the patterns, class means, and decision
hyperplane?

In [9]:
fig=go.Figure()
fig.add_trace(go.Scatter(x=NSup,y=PRT10Sup,name='Super'))
fig.add_trace(go.Scatter(x=NAvg,y=PRT10Avg,name='Average'))
fig.update_traces(marker=dict(size=10))


ixFP=np.where((y[ixAverage]!=yp[ixAverage]))[0]+50
ixFN=np.where((y[ixSuper]!=yp[ixSuper]))[0]

fig.add_trace(go.Scatter(x=X[ixFN,0],y=X[ixFN,1],name='False Average',marker_size=15,
                         marker_symbol="circle-open",marker=dict(color="red",
                                                                 line=dict(width=3,color='red'))))
fig.add_trace(go.Scatter(x=X[ixFP,0],y=X[ixFP,1],name='False Super',marker_size=15,
                         marker_symbol="circle-open",marker=dict(color="blue",
                                                                 line=dict(width=3,color='blue'))))
fig.add_trace(go.Scatter(x=muSup[0],y=muSup[1], marker_size=20,
                         marker_symbol='x',name='Super mean',
                         marker=dict(color="green",line=dict(width=1,
                                        color='black'))))
fig.add_trace(go.Scatter(x=muAvg[0],y=muAvg[1], marker_size=20,
                         marker_symbol='x',name='Average mean',
                         marker=dict(color="yellow",line=dict(width=1,
                                        color='black'))))
fig.add_trace(go.Scatter(x=(muSup[0]+muAvg[0])/2,y=(muSup[1]+muAvg[1])/2, marker_size=20,
                         marker_symbol="circle-dot",name='Mean of the means',
                         marker=dict(color="purple",line=dict(width=1,
                                        color='black'))))

fig.update_traces(mode='markers')
fig.add_trace(go.Scatter(x=[muSup[0][0],muAvg[0][0]],y=[muSup[1][0],muAvg[1][0]],
                         mode='lines', line=dict(color="black",width=4),name='Inter mean segment'))


W=((muSup-muAvg).T@Ci).T
b=-0.5*(muSup-muAvg).T@Ci@(muSup+muAvg)

x1=np.arange(20,130,1)

x2=-(W[0,0]/W[1,0])*x1-b/W[1,0]
fig.add_trace(go.Scatter(x=x1,y=x2.flatten(),
                         mode='lines', line=dict(dash='dash',color="gray",width=4),name='Hyperplane'))

fig.update_xaxes(title_text='N')
fig.update_yaxes(title_text='PRT/10')
fig.update_layout(
    autosize=False,
    width=900,
    height=800)
fig.show()

# FISHER Linear Discriminant

1. Consider again Cork-Stoppers dataset and the two-dimensional input space defined by N and PRT/10.<p>
• Generate the training and testing data sets by selecting randomly 70% of the samples for training and the remaining 30% for testing.

In [10]:
from sklearn.model_selection import train_test_split


X1 = np.array([NSup,PRT10Sup]).T 
X2 = np.array([NAvg,PRT10Avg]).T
X = np.concatenate((X1,X2),axis=0) # create a new dataset X by concatenating the feature vectors 
y = np.concatenate((CorkStopp['C'].to_numpy()[ixSuper],CorkStopp['C'].to_numpy()[ixAverage])) # labels vector

# Split the dataset into training and testing sets
nSamp = np.shape(X)[0] 
nSampTr = int(nSamp*0.7) #
rng = np.random.default_rng() 
perms = rng.permutation(nSamp)

Xtr = X[perms[0:nSampTr],:]
ytr = y[perms[0:nSampTr]]
Xte = X[perms[nSampTr:],:]
yte = y[perms[nSampTr:]]

• Create a linear classifer based on Fisher Linear Discriminant analysis and on the selected training data.

In [11]:
ixSup = np.where(ytr==1)[0]
ixAvg = np.where(ytr==2)[0]

muSup = np.array([np.mean(Xtr[ixSup,:],axis=0)]).T
muAvg = np.array([np.mean(Xtr[ixAvg,:],axis=0)]).T

S1 = (Xtr[ixSup,:].T-muSup)@(Xtr[ixSup,:].T-muSup).T
S2 = (Xtr[ixAvg,:].T-muAvg)@(Xtr[ixAvg,:].T-muAvg).T

Sw = S1+S2 # within-class scatter matrix, which is the sum of the scatter matrices for the "Super" and "Average" classes, representing the total variability within each class
SwInv = np.linalg.inv(Sw) # inverse of the within-class scatter matrix, which is used to compute the linear discriminant function for LDA by weighting the difference between the class means

w = SwInv@(muSup-muAvg) # weight vector

b = -(w.T@muSup+w.T@muAvg) / 2 # bias 

• Plot the training data and the computed hyperplane.

In [12]:
fig = go.Figure()
fig.add_trace(go.Scatter(x=Xtr[ixSup,0],y=Xtr[ixSup,1],mode='markers',name='Super'))
fig.add_trace(go.Scatter(x=Xtr[ixAvg,0],y=Xtr[ixAvg,1],mode='markers',name='Average'))
fig.update_traces(marker=dict(size=10))


x1=np.arange(20,130,1)

x2=-(w[0,0]/w[1,0])*x1 -b/w[1,0] 
fig.add_trace(go.Scatter(x=x1,y=x2.flatten(),
                         mode='lines', line=dict(dash='dash',color="gray",width=4),name='Hyperplane'))

fig.update_xaxes(title_text='N')
fig.update_yaxes(title_text='PRT/10')
fig.update_layout(
    autosize=False,
    width=900,
    height=800)
fig.show()

-Plot the testing data and the computed hyperplane.

In [13]:
ixSupTe=np.where(yte==1)[0]
ixAvgTe=np.where(yte==2)[0]


fig=go.Figure()
fig.add_trace(go.Scatter(x=Xte[ixSupTe,0],y=Xte[ixSupTe,1],mode='markers',name='Super'))
fig.add_trace(go.Scatter(x=Xte[ixAvgTe,0],y=Xte[ixAvgTe,1],mode='markers',name='Average'))
fig.update_traces(marker=dict(size=10))


x1 = np.arange(20,130,1)

x2=-(w[0,0]/w[1,0])*x1-b/w[1,0]
fig.add_trace(go.Scatter(x=x1,y=x2.flatten(),
                         mode='lines', line=dict(dash='dash',color="gray",width=4),name='Hyperplane'))

fig.update_xaxes(title_text='N')
fig.update_yaxes(title_text='PRT/10')
fig.update_layout(
    autosize=False,
    width=900,
    height=800)
fig.show()

• Compute the performance on the training and testing sets

In [14]:
yptr=np.ones(np.shape(ytr)) # vector of predicted labels for the training set
dx = (w.T@Xtr.T+b).flatten() # compute the decision function values for the training samples
yptr[dx<0]=2


TPTr=np.shape(np.where((ytr[ixSup]==yptr[ixSup]))[0])[0]
TNTr=np.shape(np.where((ytr[ixAvg]==yptr[ixAvg]))[0])[0]
FPTr=np.shape(np.where((ytr[ixAvg]!=yptr[ixAvg]))[0])[0]
FNTr=np.shape(np.where((ytr[ixSup]!=yptr[ixSup]))[0])[0]


SSTr=TPTr/(TPTr+FNTr)
SPTr=TNTr/(TNTr+FPTr)
PRTr=TPTr/(TPTr+FPTr)
F1ScoreTr=2*(PRTr*SSTr)/(PRTr+SSTr)
ACTr=(TNTr+TPTr)/(TPTr+TNTr+FPTr+FNTr)

print("-----------Training----------------")
print("Sensitivity(%)="+str(SSTr*100))
print("Specificity(%)="+str(SPTr*100))
print("Precision(%)="+str(PRTr*100))
print("F1Score(%)="+str(F1ScoreTr*100))
print("Accuracy(%)="+str(ACTr*100))


ypte = np.ones(np.shape(yte)) # vector of predicted labels for the testing set
dx = (w.T@Xte.T+b).flatten() # compute the decision function values for the testing samples
ypte[dx<0] = 2


TPTe = np.shape(np.where((yte[ixSupTe]==ypte[ixSupTe]))[0])[0]
TNTe = np.shape(np.where((yte[ixAvgTe]==ypte[ixAvgTe]))[0])[0]
FPTe = np.shape(np.where((yte[ixAvgTe]!=ypte[ixAvgTe]))[0])[0]
FNTe = np.shape(np.where((yte[ixSupTe]!=ypte[ixSupTe]))[0])[0]


SSTe = TPTe/(TPTe+FNTe)
SPTe = TNTe/(TNTe+FPTe)
PRTe = TPTe/(TPTe+FPTe)
F1ScoreTe = 2*(PRTe*SSTe)/(PRTe+SSTe)
ACTe = (TNTe+TPTe)/(TPTe+TNTe+FPTe+FNTe)

print("-----------Testing----------------")
print("Sensitivity(%)="+str(SSTe*100))
print("Specificity(%)="+str(SPTe*100))
print("Precision(%)="+str(PRTe*100))
print("F1Score(%)="+str(F1ScoreTe*100))
print("Accuracy(%)="+str(ACTe*100))

-----------Training----------------
Sensitivity(%)=93.75
Specificity(%)=84.21052631578947
Precision(%)=83.33333333333334
F1Score(%)=88.23529411764706
Accuracy(%)=88.57142857142857
-----------Testing----------------
Sensitivity(%)=100.0
Specificity(%)=91.66666666666666
Precision(%)=94.73684210526315
F1Score(%)=97.2972972972973
Accuracy(%)=96.66666666666667
